In [2]:
# from google.colab import drive
import torch
import sys
# drive.mount('/content/gdrive', force_remount=True)


#base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/Final Project"
base_dir = "/content/gdrive/MyDrive/Final Project"

sys.path.append(base_dir)

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


Device set to cuda


In [3]:
import torch.nn.functional as F

In [4]:
import importlib
import Models.GPT_Model as GPT_Model
import Datasets.DataLoader as DataLoader_Lib

importlib.reload(GPT_Model)
importlib.reload(DataLoader_Lib)

from Models.GPT_Model import GPT2_Lag, GPTConfig
from Datasets.DataLoader import TinyShakespeareDataLoader, TinyStoriesDataLoader

In [ ]:
importlib.reload(GPT_Model)
B = 4 
block_size = 128

loader = TinyStoriesDataLoader(max_length= block_size, batch_size=B)
train_loader, val_loader = loader.get_data()


Map: 100%|██████████| 10000/10000 [00:01<00:00, 7383.09 examples/s]


In [5]:
importlib.reload(GPT_Model)
B = 32 # Adjust batch size based on your Colab GPU memory
SL = 256
config = GPTConfig(num_heads = 12,
  num_layers = 12,
  vocab_size = 50257,
  embedding_dim = 768,
  block_size = SL,
  lag_behind = 1,
  dropout = .1)

model = GPT2_Lag(config, device)
model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),   # default is (0.9, 0.999)
    eps=1e-8,            # numerical stability, default is fine
    weight_decay=0.1
)

# Config = GPTConfig(num_heads = 12,
#   num_layers = 12,
#   vocab_size = 50257,
#   embedding_dim = 768,
#   block_size = block_size,
#   lag_behind = 1,
#   dropout = .1,
#   pad_token_id=loader.pad_token())

# model = GPT2_Lag(Config, device)
# model.to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [6]:
import torch
import numpy as np

class CombinedBinDataLoader:
    def __init__(self, filename, B, SL, config, split='train'):
        self.B = B
        self.SL = SL
        self.config = config
        
        # Memory-map the binary file (stays on disk, essentially 0 RAM usage)
        self.data = np.memmap(filename, dtype=np.uint16, mode='r')
        
        # Calculate the 90% split index
        n = int(0.9 * len(self.data))
        
        if split == 'train':
            self.split_data = self.data[:n]
        else:
            self.split_data = self.data[n:]
            
        self.cursor = 0
        print(f"Initialized {split} loader with {len(self.split_data):,} tokens.")

    def get_data(self):
        # 1. Fetch the continuous chunk from disk and convert to torch.long (int64)
        chunk = self.split_data[self.cursor : self.cursor + self.B * self.SL + 1]
        buf = torch.from_numpy(chunk.astype(np.int64))
        
        # 2. Standard autoregressive shifting
        x = buf[:-1].view(self.B, self.SL)
        y = buf[1:].view(self.B, self.SL)
        
        # 3. --- Y2 LAG BEHIND LOGIC ---
        k = self.config.lag_behind
        shift = k 

        # Fill with -100 (PyTorch's default ignore_index for Cross Entropy)
        y2 = torch.full_like(x, -100) 

        # Shift the sequence to the right by 'shift' steps
        if shift > 0:
            y2[:, shift:] = x[:, :-shift]
        elif shift == 0:
            y2 = x.clone()

        # 4. Advance the cursor for the next batch
        self.cursor += self.B * self.SL
        
        # Reset cursor if we hit the end of our split
        if self.cursor + (self.B * self.SL + 1) > len(self.split_data):
            self.cursor = 0
            
        return x, y, y2

In [16]:
train_loader = CombinedBinDataLoader('./Datasets/combined_dataset.bin', B, SL, config, split='train')
val_loader = CombinedBinDataLoader('./Datasets/combined_dataset.bin', B, SL, config, split='val')

Initialized train loader with 27,433,006 tokens.
Initialized val loader with 3,048,112 tokens.


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
train_loader = CombinedBinDataLoader(
    '/content/drive/MyDrive/fineweb_1B.bin',
    B, SL, config=config, split='train'
)

Initialized train loader with 900,000,157 tokens.


In [9]:
val_loader = CombinedBinDataLoader('/content/drive/MyDrive/fineweb_1B.bin', B, SL, config, split='val')

Initialized val loader with 100,000,018 tokens.


In [10]:
import math

In [11]:
@torch.no_grad()
def estimate_loss(model, loader, device, eval_iters=10):
    model.eval()
    fwd_losses = torch.zeros(eval_iters)
    bwd_losses = torch.zeros(eval_iters)

    loader.cursor = 0
    
    for k in range(eval_iters):
        x, y, _ = loader.get_data()
        x, y = x.to(device), y.to(device)

        logits_pre, _, _ = model(x, y)

        V = model.config.vocab_size
        skip_dist = model.config.lag_behind

        fwd_losses[k] = F.cross_entropy(
            logits_pre.view(-1, V), y.view(-1)
        ).item()

        # bwd_targets = x[:, :-skip_dist]
        # bwd_logits  = logits_fut[:, skip_dist:]
        # bwd_losses[k] = F.cross_entropy(
        #     bwd_logits.reshape(-1, V), bwd_targets.reshape(-1)
        # ).item()

    model.train()
    avg_fwd = fwd_losses.mean().item()
    # avg_bwd = bwd_losses.mean().item()
    avg_bwd = 0  # Placeholder since backward loss is not implemented
    ppl = torch.exp(torch.tensor(avg_fwd)).item()
    return avg_fwd, ppl, avg_bwd

def get_lr(step, num_steps_train, warmup_steps=1000, max_lr=3e-4, min_lr=3e-5):
    if step < warmup_steps:
        return max_lr * (step / warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (
        1 + math.cos(math.pi * (step - warmup_steps) / (num_steps_train - warmup_steps))
    )

tokens_per_step = train_loader.B * train_loader.SL
# def train_loop(model, optimizer, device, train_loader, val_loader,
#                num_steps_train, num_steps_val, lam=0.5):

#     print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

#     fwd_loss, ppl, bwd_loss = estimate_loss(model, val_loader, device, num_steps_val)
#     print(f"Step    0 | Val FWD: {fwd_loss:.4f} | PPL: {ppl:.2f} | Val BWD: {bwd_loss:.4f}")

#     tokens_seen = 0
#     for step in range(num_steps_train):

#         if step % 100 == 0 and step > 0:
#             fwd_loss, ppl, bwd_loss = estimate_loss(model, val_loader, device, num_steps_val)
#             print(f"Step {step:4d} | Val FWD: {fwd_loss:.4f} | PPL: {ppl:.2f} | Val BWD: {bwd_loss:.4f}")

#         lr = get_lr(step, num_steps_train)
#         for param_group in optimizer.param_groups:
#             param_group['lr'] = lr

#         x, y, _ = train_loader.get_data()
#         x, y = x.to(device), y.to(device)

#         optimizer.zero_grad()
#         _, _, loss = model(x, y, lam=lam)
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # missing
#         optimizer.step()
#         tokens_seen += tokens_per_step
#         if step % 10 == 0:
#             print(f"step {step:4d} | tokens {tokens_seen:,} | train loss {loss.item():.4f}")

from torch.cuda.amp import autocast, GradScaler

def train_loop(model, optimizer, device, train_loader, val_loader,
               num_steps_train, num_steps_val, lam=0.5):

    scaler = GradScaler()
    tokens_per_step = train_loader.B * train_loader.SL
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    history = {
        'train_steps': [],
        'train_loss':  [],
        'val_steps':   [],
        'val_loss':    [],
        'val_ppl':     []
    }

    fwd_loss, ppl, bwd_loss = estimate_loss(model, val_loader, device, num_steps_val)
    print(f"Step    0 | Val FWD: {fwd_loss:.4f} | PPL: {ppl:.2f} | Val BWD: {bwd_loss:.4f}")
    history['val_steps'].append(0)
    history['val_loss'].append(fwd_loss)
    history['val_ppl'].append(ppl)

    tokens_seen = 0
    for step in range(num_steps_train):

        if step % 100 == 0 and step > 0:
            fwd_loss, ppl, bwd_loss = estimate_loss(model, val_loader, device, num_steps_val)
            print(f"Step {step:4d} | Val FWD: {fwd_loss:.4f} | PPL: {ppl:.2f} | Val BWD: {bwd_loss:.4f}")
            history['val_steps'].append(step)
            history['val_loss'].append(fwd_loss)
            history['val_ppl'].append(ppl)

        lr = get_lr(step, num_steps_train)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        x, y, _ = train_loader.get_data()
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        with autocast():
            _, _, loss = model(x, y, lam=lam)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        tokens_seen += tokens_per_step
        if step % 10 == 0:
            history['train_steps'].append(step)
            history['train_loss'].append(loss.item())
            print(f"step {step:4d} | tokens {tokens_seen:,} | lr {lr:.2e} | train loss {loss.item():.4f}")

        if step % 500 == 0 and step > 0:
            save_checkpoint(model, optimizer, step, loss,
                f'/content/drive/MyDrive/checkpoint_step{step}.pt')

    return history

In [12]:
print(f"Train tokens: {len(train_loader.split_data):,}")
print(f"Val tokens:   {len(val_loader.split_data):,}")
print(f"Tokens per step: {train_loader.B * train_loader.SL:,}")

Train tokens: 900,000,157
Val tokens:   100,000,018
Tokens per step: 8,192


In [ ]:
!nvidia-smi

In [14]:
def save_checkpoint(model, optimizer, step, loss, path):
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, path)

def load_checkpoint(model, optimizer, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['step'], checkpoint['loss']

In [15]:
import json
num_steps_train = 50000
num_steps_val = 10
history = train_loop(model, optimizer, device, train_loader, val_loader,num_steps_train, num_steps_val)

with open('/content/drive/MyDrive/vanilla_history.json', 'w') as f:
    json.dump(history, f)

/tmp/ipython-input-4107465450.py:79: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Trainable parameters: 123,849,984
Step    0 | Val FWD: 7.6603 | PPL: 2122.46 | Val BWD: 0.0000


/tmp/ipython-input-4107465450.py:115: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


step    0 | tokens 8,192 | lr 0.00e+00 | train loss 7.5884
step   10 | tokens 90,112 | lr 3.00e-06 | train loss 8.0544
step   20 | tokens 172,032 | lr 6.00e-06 | train loss 7.5071
step   30 | tokens 253,952 | lr 9.00e-06 | train loss 7.7379
step   40 | tokens 335,872 | lr 1.20e-05 | train loss 7.8823
step   50 | tokens 417,792 | lr 1.50e-05 | train loss 7.9768
step   60 | tokens 499,712 | lr 1.80e-05 | train loss 7.8668
step   70 | tokens 581,632 | lr 2.10e-05 | train loss 7.5884
step   80 | tokens 663,552 | lr 2.40e-05 | train loss 8.0946
step   90 | tokens 745,472 | lr 2.70e-05 | train loss 7.5633
Step  100 | Val FWD: 7.6272 | PPL: 2053.28 | Val BWD: 0.0000
step  100 | tokens 827,392 | lr 3.00e-05 | train loss 7.4772
step  110 | tokens 909,312 | lr 3.30e-05 | train loss 7.5069
step  120 | tokens 991,232 | lr 3.60e-05 | train loss 7.6396
step  130 | tokens 1,073,152 | lr 3.90e-05 | train loss 7.7492
step  140 | tokens 1,155,072 | lr 4.20e-05 | train loss 7.5797
step  150 | tokens 1,23

: 

In [ ]:
from datasets import load_dataset
import tiktoken
import numpy as np
from tqdm import tqdm

In [21]:
dataset = load_dataset("HuggingFaceFW/fineweb",
                       name="sample-10BT",
                       split="train",
                       streaming=True)

enc = tiktoken.get_encoding('gpt2')
output_file = '/content/drive/MyDrive/fineweb_200M.bin'
tokens_buffer = []
total_tokens = 0
target_tokens = 1_000_000_000

with open(output_file, 'wb') as f:
    for item in tqdm(dataset):
        tokens = enc.encode_ordinary(item['text'])
        tokens.append(50256)
        tokens_buffer.extend(tokens)
        total_tokens += len(tokens)

        if len(tokens_buffer) >= 1_000_000:
            np_tokens = np.array(tokens_buffer, dtype=np.uint16)
            f.write(np_tokens.tobytes())
            tokens_buffer = []

        if total_tokens >= target_tokens:
            break

    if tokens_buffer:
        np_tokens = np.array(tokens_buffer, dtype=np.uint16)
        f.write(np_tokens.tobytes())

print(f"Saved {total_tokens:,} tokens to {output_file}")

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

1448089it [17:43, 1361.78it/s]


Saved 1,000,000,175 tokens to /content/drive/MyDrive/fineweb_200M.bin


In [18]:
import time
model.train()
x, y, _ = train_loader.get_data()
x, y = x.to(device), y.to(device)

start = time.time()
for _ in range(10):
    with torch.cuda.amp.autocast():
        _, _, loss = model(x, y)
    loss.backward()
    optimizer.zero_grad()
elapsed = time.time() - start
steps_per_sec = 10 / elapsed
print(f"Steps/sec: {steps_per_sec:.2f}")
print(f"Overnight steps (8hr): {int(steps_per_sec * 8 * 3600):,}")

/tmp/ipython-input-1441937506.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Steps/sec: 6.16
Overnight steps (8hr): 177,407


In [19]:
def save_checkpoint(model, optimizer, step, loss, path):
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, path)

def load_checkpoint(model, optimizer, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['step'], checkpoint['loss']